In [1]:
import json
import pandas as pd

BATCH_SIZE = 50
IS_DEBUG = True
DEBUG_QUESTION_SIZE = 10
LLM_MODEL = "gpt-4.1-mini"
DATASET_NAME = "hotpotQA"

In [2]:
from dotenv import load_dotenv
import os

load_dotenv()
openai_api_key = os.getenv("OPENAI_API_KEY")

## 1. Load the HotpotQA dataset

In [3]:
train_hotpot_qa_path = r'C:\Users\Sudheera\Documents\Phd\LLMs_and_KGs\hotpotQA\hotpot_train_v1.1.json'
test_hotpot_qa_path = r'C:\Users\Sudheera\Documents\Phd\LLMs_and_KGs\hotpotQA\hotpot_test_v1.1.json'

In [4]:
with open(train_hotpot_qa_path, 'r', encoding='utf-8') as f:
    train_hotpot_qa_json = json.load(f)

with open(test_hotpot_qa_path, 'r', encoding='utf-8') as f:
    test_hotpot_qa_json = json.load(f)

In [5]:
def divide_and_batch(train_json, batch_size):
    batch_context = []
    batch_question = []
    batch_answer = []

    for i in range(0, len(train_json), batch_size):
        batch = train_json[i:i + batch_size]
        context = []
        question = []
        answer = []

        for item in batch:
            context_text = ""
            j = 1
            for ctx in item['context']:
                context_text += f"Title {j} : {ctx[0]} \nParagraph {j} : {''.join(ctx[1])}\n"
                j += 1
            context.append(context_text)
            question.append(item['question'])
            answer.append(item['answer'])

        batch_context.append(context)
        batch_question.append(question)
        batch_answer.append(answer)

    return batch_context, batch_question, batch_answer

In [6]:
if IS_DEBUG:
    train_hotpot_qa_json = train_hotpot_qa_json[:DEBUG_QUESTION_SIZE]
    test_hotpot_qa_json = test_hotpot_qa_json[:DEBUG_QUESTION_SIZE]
    batch_context, batch_question, batch_answer = divide_and_batch(train_hotpot_qa_json, BATCH_SIZE)
else:
    batch_context, batch_question, batch_answer = divide_and_batch(test_hotpot_qa_json, BATCH_SIZE)

In [7]:
for i in range(len(batch_context[0])):
    print(f"Batch {i + 1}:")
    print("Context:", batch_context[0][i])
    print("Question:", batch_question[0][i])
    print("Answer:", batch_answer[0][i])
    print("-" * 50)
    if i == 9:  # Limit to 3 batches for brevity
        break

Batch 1:
Context: Title 1 : Radio City (Indian radio station) 
Paragraph 1 : Radio City is India's first private FM radio station and was started on 3 July 2001. It broadcasts on 91.1 (earlier 91.0 in most cities) megahertz from Mumbai (where it was started in 2004), Bengaluru (started first in 2001), Lucknow and New Delhi (since 2003). It plays Hindi, English and regional songs. It was launched in Hyderabad in March 2006, in Chennai on 7 July 2006 and in Visakhapatnam October 2007. Radio City recently forayed into New Media in May 2008 with the launch of a music portal - PlanetRadiocity.com that offers music related news, videos, songs, and other music-related features. The Radio station currently plays a mix of Hindi and Regional music. Abraham Thomas is the CEO of the company.
Title 2 : History of Albanian football 
Paragraph 2 : Football in Albania existed before the Albanian Football Federation (FSHF) was created. This was evidenced by the team's registration at the Balkan Cup tou

In [8]:
del train_hotpot_qa_json
del test_hotpot_qa_json

## 2. Generate LLM pipeline

In [9]:
from langchain_openai import ChatOpenAI
from langchain_core.globals import set_debug, set_verbose, set_llm_cache
from langchain_community.cache import InMemoryCache
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

set_debug(False)
set_verbose(False)
set_llm_cache(InMemoryCache())

llm_model = ChatOpenAI(model=LLM_MODEL, api_key=openai_api_key)

In [10]:
conventional_prompt_system = """
First, create a knowledge graph by extracting facts from each sentence in the given input story. Once this is
done, I will pose a complex question requiring multi-step reasoning. Decompose the question into simpler
sub-questions focusing on identifying crucial entities, their relationships, and specific details. Tackle these
sub-questions sequentially, referencing the knowledge graph for information. Connect the answers from these
sub-questions step by step, until arrive at a final answer to the initial complex question.
Provide step by step reasoning first. Then, print this heading on its own line (exactly as written):
### FINAL ANSWER
On the very next line, output ONLY the answer to the question.
Finally, end without adding anything after that word.
"""

conventional_prompt_human = """
<story>
{story}
</story>

<question>
{question}
</question>
"""

conventional_prompt = ChatPromptTemplate(
    [("system", conventional_prompt_system),
     ("human", conventional_prompt_human)]
)

conventional_chain = conventional_prompt | llm_model | StrOutputParser()

## 3. Generating batches

In [11]:
batch_input_list = []
main_df = pd.DataFrame()

for i in range(len(batch_context)):
    batch_num_list = [i] * len(batch_context[i])
    question_num_list = list(range(1, len(batch_context[i]) + 1))
    context_list = []
    question_list = []
    answers_list = []

    batch_input_item = []
    j = 0
    for batch in range(len(batch_context[i])):
        context = batch_context[i][batch]
        batch_input_item.append({
            "story": context,
            "question": batch_question[i][j]
        })
        context_list.append(context)
        question_list.append(batch_question[i][j])
        answers_list.append(batch_answer[i][j])
        j += 1
    batch_input_list.append(batch_input_item)
    df = pd.DataFrame({
        "batch_num": batch_num_list,
        "question_num": question_num_list,
        "context": context_list,
        "question": question_list,
        "answer": answers_list
    })
    main_df = pd.concat([main_df, df], ignore_index=True)

for i, batch in enumerate(batch_input_list):
    print(f"Batch {i + 1}:")
    for item in batch:
        print("story:", item['story'])
        print("question:", item['question'])
        print("-" * 50)

Batch 1:
story: Title 1 : Radio City (Indian radio station) 
Paragraph 1 : Radio City is India's first private FM radio station and was started on 3 July 2001. It broadcasts on 91.1 (earlier 91.0 in most cities) megahertz from Mumbai (where it was started in 2004), Bengaluru (started first in 2001), Lucknow and New Delhi (since 2003). It plays Hindi, English and regional songs. It was launched in Hyderabad in March 2006, in Chennai on 7 July 2006 and in Visakhapatnam October 2007. Radio City recently forayed into New Media in May 2008 with the launch of a music portal - PlanetRadiocity.com that offers music related news, videos, songs, and other music-related features. The Radio station currently plays a mix of Hindi and Regional music. Abraham Thomas is the CEO of the company.
Title 2 : History of Albanian football 
Paragraph 2 : Football in Albania existed before the Albanian Football Federation (FSHF) was created. This was evidenced by the team's registration at the Balkan Cup tourn

In [12]:
main_df

,batch_num,question_num,context,question,answer
0,0,1,Title 1 : Radio City (Indian radio station) \n...,Which magazine was started first Arthur's Maga...,Arthur's Magazine
1,0,2,Title 1 : Ritz-Carlton Jakarta \nParagraph 1 :...,The Oberoi family is part of a hotel company t...,Delhi
2,0,3,Title 1 : Lisa Simpson \nParagraph 1 : Lisa Ma...,Musician and satirist Allie Goertz wrote a son...,President Richard Nixon
3,0,4,"Title 1 : Moloch: or, This Gentile World \nPar...",What nationality was James Henry Miller's wife?,American
4,0,5,Title 1 : Cadmium chloride \nParagraph 1 : Cad...,Cadmium Chloride is slightly soluble in this c...,alcohol
5,0,6,Title 1 : Li Na \nParagraph 1 : Li Na (; ; bor...,Which tennis player won more Grand Slam titles...,Jonathan Stark
6,0,7,"Title 1 : India \nParagraph 1 : India, officia...",Which genus of moth in the world's seventh-lar...,Crambidae
7,0,8,Title 1 : Verano de Escándalo (1998) \nParagra...,Who was once considered the best kick boxer in...,Badr Hari
8,0,9,Title 1 : House of Anubis \nParagraph 1 : Hous...,"The Dutch-Belgian television series that ""Hous...",2006
9,0,10,Title 1 : Mount Panorama Circuit \nParagraph 1...,What is the length of the track where the 2013...,6.213 km long


## 4. Run the LLM pipeline to generate answers

In [13]:
conventional_str_batch = []

for i, batch in enumerate(batch_input_list):
    print(f"Batch {i + 1} ... processing {len(batch)} items")
    conventional_str_list = conventional_chain.batch(batch)
    conventional_str_batch.append(conventional_str_list)

Batch 1 ... processing 10 items


In [14]:
for i, conventional_str_list in enumerate(conventional_str_batch):
    print(f"Batch {i + 1} ... processed {len(conventional_str_list)} items")
    for j, conventional_str in enumerate(conventional_str_list):
        print(f"Item {j + 1}:")
        print(conventional_str)
        print("-" * 50)

Batch 1 ... processed 10 items
Item 1:
Let's extract facts from the story to form a knowledge graph:

**Knowledge Graph Extraction:**

- Arthur's Magazine:
  - An American literary periodical.
  - Published in Philadelphia in the 19th century.
  - Edited by T.S. Arthur.
  - Existed from 1844–1846.
  - Started in 1844.

- First for Women:
  - A woman's magazine.
  - Published by Bauer Media Group in the USA.
  - Started in 1989.
  - Based in Englewood Cliffs, New Jersey.

**Complex Question Decomposition:**
"Which magazine was started first Arthur's Magazine or First for Women?"

**Sub-questions:**
1. When was Arthur's Magazine started?
2. When was First for Women started?
3. Which date is earlier?

**Step-by-step Reasoning:**

1. Arthur's Magazine was started in 1844 (from the knowledge graph: "Arthur's Magazine (1844–1846)").
2. First for Women was started in 1989 (from the knowledge graph: "The magazine was started in 1989").
3. 1844 is much earlier than 1989.

Thus, Arthur's Magazin

In [15]:
conventional_str_df = pd.DataFrame({
    "batch_num": [],
    "question_num": [],
    "answer_str": []
})

for i, conventional_str_list in enumerate(conventional_str_batch):
    batch_num_list = [i] * len(conventional_str_list)
    question_num_list = list(range(1, len(conventional_str_list) + 1))
    conventional_str_df = pd.concat([conventional_str_df, pd.DataFrame({
        "batch_num": batch_num_list,
        "question_num": question_num_list,
        "answer_str": conventional_str_list
    })], ignore_index=True)

# del conventional_str_batch
# del conventional_str_list
# del batch_input_list
# del batch_context
# del batch_question
# del batch_answer

conventional_str_df

,batch_num,question_num,answer_str
0,0.0,1.0,Let's extract facts from the story to form a k...
1,0.0,2.0,"Step by step reasoning:\n\n1. First, from Para..."
2,0.0,3.0,Let's begin by creating a knowledge graph by e...
3,0.0,4.0,Step by step reasoning:\n\n1. Identify the rel...
4,0.0,5.0,Step by Step Reasoning:\n\n1. The question is ...
5,0.0,6.0,"First, let's extract and organize the facts re..."
6,0.0,7.0,Let me start by extracting a knowledge graph f...
7,0.0,8.0,Let's start by extracting facts from each sent...
8,0.0,9.0,"First, create a knowledge graph by extracting ..."
9,0.0,10.0,Knowledge Graph Extraction:\n\n1. Mount Panora...


In [16]:
import re

def extract_final_answer(row):
    # Regex pattern to find the final answer
    final_answer_pat = r'### FINAL ANSWER\s*(.*)'
    match = re.search(final_answer_pat, row['answer_str'], re.DOTALL)
    return match.group(1).strip() if match else "No answer found"

conventional_str_df['llm_answer'] = conventional_str_df.apply(extract_final_answer, axis=1)
conventional_str_df

,batch_num,question_num,answer_str,llm_answer
0,0.0,1.0,Let's extract facts from the story to form a k...,Arthur's Magazine
1,0.0,2.0,"Step by step reasoning:\n\n1. First, from Para...",Delhi
2,0.0,3.0,Let's begin by creating a knowledge graph by e...,President Richard Nixon's middle name
3,0.0,4.0,Step by step reasoning:\n\n1. Identify the rel...,American
4,0.0,5.0,Step by Step Reasoning:\n\n1. The question is ...,Ethanol
5,0.0,6.0,"First, let's extract and organize the facts re...",Jonathan Stark
6,0.0,7.0,Let me start by extracting a knowledge graph f...,"Nepita, Indogrammodes"
7,0.0,8.0,Let's start by extracting facts from each sent...,Badr Hari
8,0.0,9.0,"First, create a knowledge graph by extracting ...",2006
9,0.0,10.0,Knowledge Graph Extraction:\n\n1. Mount Panora...,6.213 km


In [17]:
main_df = main_df.merge(conventional_str_df, on=['batch_num', 'question_num'], how='left')
main_df

,batch_num,question_num,context,question,answer,answer_str,llm_answer
0,0,1,Title 1 : Radio City (Indian radio station) \n...,Which magazine was started first Arthur's Maga...,Arthur's Magazine,Let's extract facts from the story to form a k...,Arthur's Magazine
1,0,2,Title 1 : Ritz-Carlton Jakarta \nParagraph 1 :...,The Oberoi family is part of a hotel company t...,Delhi,"Step by step reasoning:\n\n1. First, from Para...",Delhi
2,0,3,Title 1 : Lisa Simpson \nParagraph 1 : Lisa Ma...,Musician and satirist Allie Goertz wrote a son...,President Richard Nixon,Let's begin by creating a knowledge graph by e...,President Richard Nixon's middle name
3,0,4,"Title 1 : Moloch: or, This Gentile World \nPar...",What nationality was James Henry Miller's wife?,American,Step by step reasoning:\n\n1. Identify the rel...,American
4,0,5,Title 1 : Cadmium chloride \nParagraph 1 : Cad...,Cadmium Chloride is slightly soluble in this c...,alcohol,Step by Step Reasoning:\n\n1. The question is ...,Ethanol
5,0,6,Title 1 : Li Na \nParagraph 1 : Li Na (; ; bor...,Which tennis player won more Grand Slam titles...,Jonathan Stark,"First, let's extract and organize the facts re...",Jonathan Stark
6,0,7,"Title 1 : India \nParagraph 1 : India, officia...",Which genus of moth in the world's seventh-lar...,Crambidae,Let me start by extracting a knowledge graph f...,"Nepita, Indogrammodes"
7,0,8,Title 1 : Verano de Escándalo (1998) \nParagra...,Who was once considered the best kick boxer in...,Badr Hari,Let's start by extracting facts from each sent...,Badr Hari
8,0,9,Title 1 : House of Anubis \nParagraph 1 : Hous...,"The Dutch-Belgian television series that ""Hous...",2006,"First, create a knowledge graph by extracting ...",2006
9,0,10,Title 1 : Mount Panorama Circuit \nParagraph 1...,What is the length of the track where the 2013...,6.213 km long,Knowledge Graph Extraction:\n\n1. Mount Panora...,6.213 km


## 5. Calculate accuracy

In [18]:
system_msg = """
<role>
    You are a meticulous LLM answer evaluator. Your task is to determine if the provided llm_answer correctly matches the correct_answer for a given question.
</role>

<behavior>
    <rule name="case-insensitive-match">
        Treat answers as correct even if their case (uppercase/lowercase) does not match, as long as the content matches.
    </rule>
    <rule name="synonyms-acceptable">
        Accept synonyms, short forms, or equivalent factual answers (e.g., "alcohol" and "ethanol") as correct, unless there is a clear difference in meaning or context.
    </rule>
    <rule name="factual-containment">
        Accept answers that contain the correct answer as a substring, or are paraphrased, provided no contradictory information is introduced.
    </rule>
    <rule name="insufficient-data">
        Mark as incorrect if llm_answer indicates "insufficient data" or similar phrases, but a correct factual answer is actually provided in correct_answer.
    </rule>
    <rule name="incorrect-content">
        Mark as incorrect if the llm_answer gives a wrong, contradictory, or irrelevant response compared to correct_answer.
    </rule>
    <rule name="format-neutrality">
        Do not penalize for minor differences in format, such as punctuation or extra explanatory words, as long as the meaning is unchanged.
    </rule>
    <rule name="numerical-tolerance">
        For numerical answers, accept equivalent formats (e.g., "2006" and "September 2006" are correct if both indicate the correct year), but mark as incorrect if the core value is wrong.
    </rule>
</behavior>

<format>
1) Output your reasoning step by step, referencing the question, correct_answer, and llm_answer.
2) Clearly state if the llm_answer matches the correct_answer according to the behavior rules above.
3) End your output with the heading "### FINAL_ANSWER" followed by TRUE if the llm_answer is correct or FALSE if incorrect.
</format>
"""

human_msg = """
<question>
{question}
</question>

<correct_answer>
{correct_answer}
</correct_answer>

<llm_answer>
{llm_answer}
</llm_answer>
"""

prompt = ChatPromptTemplate([("system", system_msg), ("human", human_msg)])

In [19]:
chain = prompt | llm_model | StrOutputParser()

In [20]:
# preparing the input for the evaluation chain
batch_input_list = []

for i in range(0, main_df.shape[0], BATCH_SIZE):
    batch_df = main_df.iloc[i:i+BATCH_SIZE]
    batch_input_item = []

    for i, (index, row) in enumerate(batch_df.iterrows()):
        batch_input_item.append({
            "question": row['question'],
            "correct_answer": row['answer'],
            "llm_answer": row['llm_answer']
        })
    batch_input_list.append(batch_input_item)

for i, batch in enumerate(batch_input_list):
    print(f"Batch {i + 1}:")
    for item in batch:
        print("Question:", item['question'])
        print("Correct Answer:", item['correct_answer'])
        print("LLM Answer:", item['llm_answer'])
        print("-" * 50)

Batch 1:
Question: Which magazine was started first Arthur's Magazine or First for Women?
Correct Answer: Arthur's Magazine
LLM Answer: Arthur's Magazine
--------------------------------------------------
Question: The Oberoi family is part of a hotel company that has a head office in what city?
Correct Answer: Delhi
LLM Answer: Delhi
--------------------------------------------------
Question: Musician and satirist Allie Goertz wrote a song about the "The Simpsons" character Milhouse, who Matt Groening named after who?
Correct Answer: President Richard Nixon
LLM Answer: President Richard Nixon's middle name
--------------------------------------------------
Question:  What nationality was James Henry Miller's wife?
Correct Answer: American
LLM Answer: American
--------------------------------------------------
Question: Cadmium Chloride is slightly soluble in this chemical, it is also called what?
Correct Answer: alcohol
LLM Answer: Ethanol
--------------------------------------------

In [21]:
answer_check_str_batch = []

for i, batch in enumerate(batch_input_list):
    print(f"Batch {i + 1} ... processing {len(batch)} items")
    answer_check_str_list = chain.batch(batch)
    answer_check_str_batch.append(answer_check_str_list)

Batch 1 ... processing 10 items


In [22]:
for i, answer_check_str_list in enumerate(answer_check_str_batch):
    print(f"Batch {i + 1} ... processed {len(answer_check_str_list)} items")
    for j, answer_check_str in enumerate(answer_check_str_list):
        print(f"Item {j + 1}:")
        print(answer_check_str)
        print("-" * 50)

Batch 1 ... processed 10 items
Item 1:
1) The question asks which magazine was started first: Arthur's Magazine or First for Women?
2) The correct_answer given is "Arthur's Magazine."
3) The llm_answer is also "Arthur's Magazine."
4) According to the rules:
    - The content of the llm_answer exactly matches the correct_answer with no case or format discrepancies (format-neutrality, case-insensitive-match).
    - There is no irrelevant or contradictory information (incorrect-content).
    - The answer directly identifies the correct magazine, matching the correct factual answer (factual-containment).
5) Therefore, the llm_answer matches the correct_answer.

### FINAL_ANSWER
TRUE
--------------------------------------------------
Item 2:
1) The question asks for the city where the Oberoi hotel company's head office is located.  
2) The correct_answer is "Delhi."  
3) The llm_answer also states "Delhi."  
4) According to behavior rules, there are no issues with case, format, or synonyms 

In [23]:
# Create a DataFrame to store the evaluation results
answer_check_df = pd.DataFrame({
    "batch_num": [],
    "question_num": [],
    "answer_check_str": []
})

for i, answer_check_str_list in enumerate(answer_check_str_batch):
    batch_num_list = [i] * len(answer_check_str_list)
    question_num_list = list(range(1, len(answer_check_str_list) + 1))
    answer_check_df = pd.concat([answer_check_df, pd.DataFrame({
        "batch_num": batch_num_list,
        "question_num": question_num_list,
        "answer_check_str": answer_check_str_list
    })], ignore_index=True)

In [24]:
answer_check_df

,batch_num,question_num,answer_check_str
0,0.0,1.0,1) The question asks which magazine was starte...
1,0.0,2.0,1) The question asks for the city where the Ob...
2,0.0,3.0,1) Let's examine the question: It asks about w...
3,0.0,4.0,"1) The question is: ""What nationality was Jame..."
4,0.0,5.0,1) The question asks for the alternative/commo...
5,0.0,6.0,"1) The question asks which tennis player, Henr..."
6,0.0,7.0,1) Step-by-step reasoning:\n\n- The question a...
7,0.0,8.0,1) The question asks for the name of the kickb...
8,0.0,9.0,1) The question asks for the year the Dutch-Be...
9,0.0,10.0,1) The question asks for the length of the tra...


In [25]:
# Extract is_correct from the answer_check_str
def extract_is_correct(row):
    # Regex pattern to find the final answer
    final_answer_pat = r'### FINAL_ANSWER\s*(TRUE|FALSE)'
    match = re.search(final_answer_pat, row['answer_check_str'], re.DOTALL)
    return match.group(1).strip() if match else False

# Merge the llm_answer_str_df with main_df to get the final answers
answer_check_df['is_correct'] = answer_check_df.apply(extract_is_correct, axis=1)
main_df = main_df.merge(answer_check_df, on=['batch_num', 'question_num'], how='left')
main_df

,batch_num,question_num,context,question,answer,answer_str,llm_answer,answer_check_str,is_correct
0,0,1,Title 1 : Radio City (Indian radio station) \n...,Which magazine was started first Arthur's Maga...,Arthur's Magazine,Let's extract facts from the story to form a k...,Arthur's Magazine,1) The question asks which magazine was starte...,TRUE
1,0,2,Title 1 : Ritz-Carlton Jakarta \nParagraph 1 :...,The Oberoi family is part of a hotel company t...,Delhi,"Step by step reasoning:\n\n1. First, from Para...",Delhi,1) The question asks for the city where the Ob...,TRUE
2,0,3,Title 1 : Lisa Simpson \nParagraph 1 : Lisa Ma...,Musician and satirist Allie Goertz wrote a son...,President Richard Nixon,Let's begin by creating a knowledge graph by e...,President Richard Nixon's middle name,1) Let's examine the question: It asks about w...,TRUE
3,0,4,"Title 1 : Moloch: or, This Gentile World \nPar...",What nationality was James Henry Miller's wife?,American,Step by step reasoning:\n\n1. Identify the rel...,American,"1) The question is: ""What nationality was Jame...",TRUE
4,0,5,Title 1 : Cadmium chloride \nParagraph 1 : Cad...,Cadmium Chloride is slightly soluble in this c...,alcohol,Step by Step Reasoning:\n\n1. The question is ...,Ethanol,1) The question asks for the alternative/commo...,TRUE
5,0,6,Title 1 : Li Na \nParagraph 1 : Li Na (; ; bor...,Which tennis player won more Grand Slam titles...,Jonathan Stark,"First, let's extract and organize the facts re...",Jonathan Stark,"1) The question asks which tennis player, Henr...",TRUE
6,0,7,"Title 1 : India \nParagraph 1 : India, officia...",Which genus of moth in the world's seventh-lar...,Crambidae,Let me start by extracting a knowledge graph f...,"Nepita, Indogrammodes",1) Step-by-step reasoning:\n\n- The question a...,TRUE
7,0,8,Title 1 : Verano de Escándalo (1998) \nParagra...,Who was once considered the best kick boxer in...,Badr Hari,Let's start by extracting facts from each sent...,Badr Hari,1) The question asks for the name of the kickb...,TRUE
8,0,9,Title 1 : House of Anubis \nParagraph 1 : Hous...,"The Dutch-Belgian television series that ""Hous...",2006,"First, create a knowledge graph by extracting ...",2006,1) The question asks for the year the Dutch-Be...,TRUE
9,0,10,Title 1 : Mount Panorama Circuit \nParagraph 1...,What is the length of the track where the 2013...,6.213 km long,Knowledge Graph Extraction:\n\n1. Mount Panora...,6.213 km,1) The question asks for the length of the tra...,TRUE


In [26]:
# Calculate the accuracy
accuracy = main_df['is_correct'].value_counts(normalize=True).get('TRUE', 0) * 100
print(f"Accuracy: {accuracy:.2f}%")

Accuracy: 100.00%


In [ ]:
# GPT 4.1 - Accuracy: 100.00%